In [1]:
print("ok")

ok


In [2]:
%pwd

'/home/mulandi/Documents/programming_files/hackerthon/plp/hakathon_2/skin_diagnosis/backend/chatbot/research'

In [3]:
%ls

trials.ipynb


In [5]:
import os 
os.chdir("../")
%pwd

'/home/mulandi/Documents/programming_files/hackerthon/plp/hakathon_2/skin_diagnosis/backend/chatbot'

In [6]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import CharacterTextSplitter

/home/mulandi/anaconda3/envs/hope-chatbot/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Extract text from PDF files in a directory
def load_pdf_files(data):
 loader = DirectoryLoader(data,
 glob="*.pdf",
 loader_cls=PyPDFLoader
 )

 documents = loader.load()
 return documents

In [ ]:
extracted_data = load_pdf_files("data")

In [ ]:
extracted_data

In [ ]:
len(extracted_data)

In [ ]:
from typing import List 
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
 # Filter documents to retain only minimal necessary information.

 minimal_docs: List[Document] = [] 
 for doc in docs:
  src = doc.metadata.get("source")
  minimal_docs.append(
   Document(
    page_content=doc.page_content,
    metadata={"source": src}
   )
  )
 return minimal_docs

In [ ]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [ ]:
minimal_docs

In [ ]:
# Split the documents into smaller chunks

# import recursive character text splitter 
from langchain.text_splitter import RecursiveCharacterTextSplitter

def text_split(minimal_docs):
 text_splitter = RecursiveCharacterTextSplitter(
  chunk_size=500,
  chunk_overlap=20,
 )
 texts_chunk = text_splitter.split_documents(minimal_docs)
 return texts_chunk

In [ ]:
texts_chunk = text_split(minimal_docs)
print(f"Number of text chunks: {len(texts_chunk)}")

In [ ]:
texts_chunk = text_split(minimal_docs)
print(f"Number of text chunks: {len(texts_chunk)}")

In [ ]:
texts_chunk

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings 

def download_embeddings():
 # download the embeddings model and return the HuggingFace embeddings model.

 model_name = "sentence-transformers/all-MiniLM-L6-v2"
 embeddings = HuggingFaceEmbeddings(
  model_name=model_name
 )
 return embeddings

embedding = download_embeddings()

In [ ]:
embedding

In [ ]:
vector = embedding.embed_query("Hello world")
vector

In [ ]:
print("Vector length: ", len(vector))

In [ ]:
from dotenv import load_dotenv
import os 
load_dotenv()

In [ ]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [ ]:
from pinecone import Pinecone 
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [ ]:
pc

In [ ]:
from pinecone import ServerlessSpec

index_name = "skin-diagnosis-chatbot"

if not pc.has_index(index_name):
 pc.create_index(
  name = index_name,
  dimension = 384, # dimension of the embedding model
  metric = "cosine", # cosine similarity metric
  spec=ServerlessSpec(cloud="aws", region="us-east-1")
 )
index = pc.get_index(index_name)

In [ ]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
 documents=texts_chunk, 
 embedding=embedding,
 index_name=index_name
)

In [ ]:
# Load Existing index 

from langchain_pinecone import PineconeVectorStore

# Embed each chunk and upsert the embeddings into your Pinecone index. 
docsearch = PineconeVectorStore.from_existing_index(
 index_name=index_name,
 embedding=embedding
)

# Add more data to the existing Pinecone index

In [ ]:
dswith = Document(
 page_content="dswithbappy is a youtube channel that provide tutorials on various topics.", 
 metadata={"source": "youtube"}
)

In [ ]:
docsearch.add_documents(documents=[dswith])

In [ ]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [ ]:
retrieved_docs = retriever.invoke("What is Acne?")
retrieved_docs

In [ ]:
from langchain_openai import ChatOpenAI
chatModel = ChatOpenAI(model="gpt-4o")

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate 

In [ ]:
system_prompt = (
 "You are a helpful AI assistant that provides accurate and concise information about skin diseases based on the provided context."
 "Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer."
 "Use three sentences maximum and keep the answer concise."
 "\n\n"
 "{context}"
)

prompt = ChatPromptTemplate.from_messages(
 [
  ("system", system_prompt),
  ("human", "{input}"),
 ]
)

In [ ]:
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [ ]:
response = rag_chain.invoke({"input": "what is Acromegaly and gigantism"})
print(response["answer"])


In [ ]:
response = rag_chain.invoke({"input": "what is Acne?"})
print(response["answer"])

In [ ]:
response = rag_chain.invoke({"input": "what is the Treatment of Acne?"})
print(response["answer"])